In [ ]:
# Traffic Demand Prediction
# Blend of Day-48 lookup ladder (with spatial neighbor fallback)
# and a LightGBM structure model.

In [ ]:
import numpy as np, pandas as pd, pygeohash as pgh, lightgbm as lgb
from sklearn.neighbors import KDTree
from sklearn.metrics import r2_score

In [ ]:
"""Data loading, cleaning, geohash decoding, Day-48 aggregates, hist features, lookup ladder.

All historical features come from DAY 48 only. Day-48 training rows get
leave-one-out (LOO) aggregates; Day-49 and test rows get plain Day-48 means
from the identical source, so training mirrors inference.
"""
from __future__ import annotations
import functools
import numpy as np
import pandas as pd
import pygeohash as pgh

DATA_DIR = "dataset"

CAT_COLS = ["RoadType", "LargeVehicles", "Landmarks", "Weather"]
CAT_MAPS = {
    "RoadType":      {"Residential": 0, "Street": 1, "Highway": 2, "Unknown": 3},
    "LargeVehicles": {"Not Allowed": 0, "Allowed": 1, "Unknown": 2},
    "Landmarks":     {"No": 0, "Yes": 1, "Unknown": 2},
    "Weather":       {"Sunny": 0, "Rainy": 1, "Foggy": 2, "Snowy": 3, "Unknown": 4},
}

# hist_nb (neighbor prior) is added by src.spatial; it is part of FEATURES so
# validate/submit must call spatial.add_neighbor_feature before selecting FEATURES.
FEATURES = [
    "lat", "lon", "minutes", "min_sin", "min_cos",
    "NumberofLanes", "Temperature",
    "RoadType", "LargeVehicles", "Landmarks", "Weather",
    "hist_g", "hist_t", "hist_g5", "hist_nb",
]
# hist_gt (exact self lookup) is NOT a GBM feature (LOO undefined for Day-48
# rows). It is used only by the lookup ladder, leak-free for Day-49/test.


@functools.lru_cache(maxsize=None)
def decode_latlon(gh: str):
    ll = pgh.decode(gh)
    return ll.latitude, ll.longitude


def _parse_minutes(ts: pd.Series) -> pd.Series:
    parts = ts.str.split(":", expand=True).astype(int)
    return parts[0] * 60 + parts[1]


def load_raw():
    train = pd.read_csv(f"{DATA_DIR}/train.csv")
    test = pd.read_csv(f"{DATA_DIR}/test.csv")
    return train, test


def basic_clean(df: pd.DataFrame, temp_median: float) -> pd.DataFrame:
    """Add minutes/cyclical/geo features, impute, encode categoricals.
    temp_median is computed once from TRAIN and reused for test."""
    df = df.copy()
    df["minutes"] = _parse_minutes(df["timestamp"])
    df["min_sin"] = np.sin(2 * np.pi * df["minutes"] / 1440.0)
    df["min_cos"] = np.cos(2 * np.pi * df["minutes"] / 1440.0)
    df["g5"] = df["geohash"].str[:5]
    latlon = df["geohash"].map(decode_latlon)
    df["lat"] = latlon.map(lambda t: t[0]).astype(float)
    df["lon"] = latlon.map(lambda t: t[1]).astype(float)
    df["Temperature"] = df["Temperature"].fillna(temp_median)
    df["NumberofLanes"] = df["NumberofLanes"].fillna(0).astype(int)
    for c in CAT_COLS:
        # unseen categories -> Unknown code (never NaN, so .astype(int) is safe)
        df[c] = (df[c].fillna("Unknown").map(CAT_MAPS[c])
                 .fillna(CAT_MAPS[c]["Unknown"]).astype(int))
    return df


def build_day48_aggregates(train_clean: pd.DataFrame) -> dict:
    """Sum/count tables from DAY 48 only, for hist features + LOO + lookup."""
    d48 = train_clean[train_clean["day"] == 48]
    agg = {
        "global_mean": float(d48["demand"].mean()),
        "gt": d48.groupby(["geohash", "timestamp"])["demand"].mean(),
        "g_sum": d48.groupby("geohash")["demand"].sum(),
        "g_cnt": d48.groupby("geohash")["demand"].count(),
        "t_sum": d48.groupby("timestamp")["demand"].sum(),
        "t_cnt": d48.groupby("timestamp")["demand"].count(),
        "g5_sum": d48.groupby("g5")["demand"].sum(),
        "g5_cnt": d48.groupby("g5")["demand"].count(),
    }
    agg["g_mean"] = agg["g_sum"] / agg["g_cnt"]
    agg["t_mean"] = agg["t_sum"] / agg["t_cnt"]
    agg["g5_mean"] = agg["g5_sum"] / agg["g5_cnt"]
    return agg


def _loo_mean(keys, demand, key_sum, key_cnt, fallback):
    s = key_sum.reindex(keys).to_numpy()
    c = key_cnt.reindex(keys).to_numpy()
    out = np.where(c > 1, (s - demand) / np.maximum(c - 1, 1), np.nan)
    return pd.Series(out).fillna(fallback).to_numpy()


def add_hist_features(df: pd.DataFrame, agg: dict, is_day48_train: bool) -> pd.DataFrame:
    """Attach hist_g, hist_t, hist_g5. LOO for Day-48 training rows; plain
    Day-48 means for Day-49/test rows."""
    df = df.copy()
    gm = agg["global_mean"]
    if is_day48_train:
        d = df["demand"].to_numpy()
        df["hist_g"]  = _loo_mean(df["geohash"], d, agg["g_sum"],  agg["g_cnt"],  gm)
        df["hist_t"]  = _loo_mean(df["timestamp"], d, agg["t_sum"], agg["t_cnt"], gm)
        df["hist_g5"] = _loo_mean(df["g5"], d, agg["g5_sum"], agg["g5_cnt"], gm)
    else:
        df["hist_g"]  = df["geohash"].map(agg["g_mean"]).fillna(
                        df["g5"].map(agg["g5_mean"])).fillna(gm)
        df["hist_t"]  = df["timestamp"].map(agg["t_mean"]).fillna(gm)
        df["hist_g5"] = df["g5"].map(agg["g5_mean"]).fillna(gm)
    return df


def lookup_pred(df: pd.DataFrame, agg: dict, neighbor=None) -> np.ndarray:
    """Lookup ladder:
      1. exact (geohash, timestamp) Day-48 demand
      2. neighbor mean at that time (if `neighbor` array supplied by src.spatial)
      3. geohash daily mean -> 4. g5 prefix mean -> 5. time mean -> 6. global mean
    Leak-free for Day-49/test rows."""
    gm = agg["global_mean"]
    gt = agg["gt"]
    keys = list(zip(df["geohash"], df["timestamp"]))
    p = pd.Series([gt.get(k, np.nan) for k in keys], index=df.index)
    if neighbor is not None:
        p = p.fillna(pd.Series(neighbor, index=df.index))
    p = p.fillna(df["geohash"].map(agg["g_mean"]))
    p = p.fillna(df["g5"].map(agg["g5_mean"]))
    p = p.fillna(df["timestamp"].map(agg["t_mean"]))
    p = p.fillna(gm)
    return np.clip(p.to_numpy(), 0.0, 1.0)


def prepare(train_raw: pd.DataFrame, test_raw: pd.DataFrame):
    """Full prep: returns (train_clean, test_clean, agg, temp_median)."""
    temp_median = float(train_raw["Temperature"].median())
    train_clean = basic_clean(train_raw, temp_median)
    test_clean = basic_clean(test_raw, temp_median)
    agg = build_day48_aggregates(train_clean)
    return train_clean, test_clean, agg, temp_median

In [ ]:
"""Spatial neighbor averaging. Rescues sparse / unseen (geohash, time) cells.

Ladder inside neighbor_predict: nearest Day-48 neighbors at the SAME minute;
if none have a reading (data is ~58% sparse), widen to +/- `window` minutes.
Self is always excluded -> leak-free for Day-48 training rows too.
"""
from __future__ import annotations
import numpy as np
from sklearn.neighbors import KDTree


def build_spatial_index(train_clean, k=12):
    """Index built from DAY 48 only (the reference day, same source as test)."""
    d48 = train_clean[train_clean["day"] == 48]
    geos = d48["geohash"].unique()
    coords = np.array([decode_latlon(g) for g in geos])
    tree = KDTree(coords)
    gt = d48.groupby(["geohash", "minutes"])["demand"].mean()
    gminutes, gdemand = {}, {}
    for g, sub in d48.groupby("geohash"):
        gminutes[g] = sub["minutes"].to_numpy()
        gdemand[g] = sub["demand"].to_numpy()
    return {"geos": geos, "coords": coords, "tree": tree, "gt": gt,
            "gminutes": gminutes, "gdemand": gdemand, "k": k}


def neighbor_predict(df, sp, window=45):
    """Per-row mean demand of the k nearest Day-48 geohashes (excluding self)
    at the same minute; widen to +/- window minutes if the exact minute is empty.
    Returns array with np.nan where even the window has no neighbor data."""
    k = sp["k"]
    geos = sp["geos"]; gt = sp["gt"]
    gmin = sp["gminutes"]; gdem = sp["gdemand"]
    ug = df["geohash"].unique()
    ucoords = np.array([decode_latlon(g) for g in ug])
    # query k+1 to allow dropping self
    _, idx = sp["tree"].query(ucoords, k=min(k + 1, len(geos)))
    nb_map = {}
    for i, g in enumerate(ug):
        cand = [geos[j] for j in idx[i] if geos[j] != g][:k]
        nb_map[g] = cand
    out = np.full(len(df), np.nan)
    gvals = df["geohash"].to_numpy(); mvals = df["minutes"].to_numpy()
    for r in range(len(df)):
        g = gvals[r]; m = int(mvals[r])
        nbs = nb_map.get(g, [])
        vals = [gt.get((nb, m), np.nan) for nb in nbs]
        vals = [v for v in vals if v == v]
        if not vals:
            for nb in nbs:
                mm = gmin.get(nb); 
                if mm is None:
                    continue
                sel = np.abs(mm - m) <= window
                if sel.any():
                    vals.append(float(gdem[nb][sel].mean()))
        if vals:
            out[r] = float(np.mean(vals))
    return out


def add_neighbor_feature(df, sp, agg, window=45):
    """Add hist_nb column; fill remaining gaps with hist_g then global mean."""
    df = df.copy()
    nb = neighbor_predict(df, sp, window=window)
    # use .to_numpy() (positional) to avoid pandas index-alignment footguns
    hist_g = df["hist_g"].to_numpy() if "hist_g" in df.columns else np.full(len(df), np.nan)
    nb = np.where(np.isnan(nb), hist_g, nb)
    df["hist_nb"] = np.where(np.isnan(nb), agg["global_mean"], nb)
    return df

In [ ]:
"""LightGBM training + prediction with optional log1p target transform."""
from __future__ import annotations
import numpy as np
import lightgbm as lgb

PARAMS = dict(
    objective="regression",
    metric="rmse",
    learning_rate=0.03,
    num_leaves=63,
    min_child_samples=40,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    verbose=-1,
)


def train_gbm(X, y, cat_idx, eval_X=None, eval_y=None,
              use_log=True, num_boost_round=3000):
    """Train LightGBM. With an eval set: early-stop and return best_iteration;
    else train full num_boost_round. Returns (booster, best_iteration)."""
    yt = np.log1p(y) if use_log else y
    dtrain = lgb.Dataset(X, label=yt, categorical_feature=cat_idx)
    callbacks = [lgb.log_evaluation(period=0)]
    valid_sets = None
    if eval_X is not None:
        eyt = np.log1p(eval_y) if use_log else eval_y
        dvalid = lgb.Dataset(eval_X, label=eyt, categorical_feature=cat_idx,
                             reference=dtrain)
        valid_sets = [dvalid]
        callbacks.append(lgb.early_stopping(stopping_rounds=100, verbose=False))
    booster = lgb.train(PARAMS, dtrain, num_boost_round=num_boost_round,
                        valid_sets=valid_sets, callbacks=callbacks)
    best_iter = booster.best_iteration or num_boost_round
    return booster, best_iter


def predict_gbm(booster, X, use_log=True):
    raw = booster.predict(X, num_iteration=booster.best_iteration or None)
    pred = np.expm1(raw) if use_log else raw
    return np.clip(pred, 0.0, 1.0)

In [ ]:
"""Train final model and write submission CSVs.
Trains on Day 48 by default; optionally adds Day 49 if TRAIN_ON_DAY49=True.
Set BEST_ITER / W_LOOKUP / USE_NEIGHBOR_LOOKUP_FOR_BLEND from validate.py first.
"""
import numpy as np
import pandas as pd

# --- set these from validate.py output ---
BEST_ITER = 245         # from validate.py best_iteration
W_LOOKUP = 0.0          # from validate.py w_lookup
USE_LOG = True
USE_NEIGHBOR_LOOKUP_FOR_BLEND = False  # validate.py printed lookup=plain
TRAIN_ON_DAY49 = False  # IMPORTANT: Day-49 rows are morning-only and `day` is
#   not a feature, so adding them biases the model toward 0:00-2:00 and can hurt
#   the 2:15-13:45 test window. Keep False unless leaderboard feedback says otherwise.


def main():
    tr, te = load_raw()
    trc, tec, agg, _tm = prepare(tr, te)
    sp = build_spatial_index(trc, k=12)

    d48 = add_neighbor_feature(add_hist_features(trc[trc["day"] == 48].copy(), agg, True), sp, agg)
    test = add_neighbor_feature(add_hist_features(tec.copy(), agg, False), sp, agg)
    if TRAIN_ON_DAY49:
        d49 = add_neighbor_feature(add_hist_features(trc[trc["day"] == 49].copy(), agg, False), sp, agg)
        train_df = pd.concat([d48, d49], ignore_index=True)
    else:
        train_df = d48

    cat_idx = [FEATURES.index(c) for c in CAT_COLS]
    booster, _ = train_gbm(train_df[FEATURES], train_df["demand"].to_numpy(),
                           cat_idx, use_log=USE_LOG, num_boost_round=BEST_ITER)

    pred_gbm = predict_gbm(booster, test[FEATURES], use_log=USE_LOG)
    pred_lookup = lookup_pred(test, agg)                      # plain ladder (primary)
    nb_test = neighbor_predict(test, sp)
    pred_lookup_nb = lookup_pred(test, agg, neighbor=nb_test) # alt candidate
    # blend with the SAME lookup variant validate.py selected (see flag above)
    lookup_for_blend = pred_lookup_nb if USE_NEIGHBOR_LOOKUP_FOR_BLEND else pred_lookup
    pred_blend = np.clip(W_LOOKUP * lookup_for_blend + (1 - W_LOOKUP) * pred_gbm, 0.0, 1.0)

    def write(name, preds):
        preds = np.asarray(preds, dtype=float)
        out = pd.DataFrame({"Index": tec["Index"].to_numpy(), "demand": preds})
        assert out.shape == (41778, 2), f"bad shape {out.shape}"
        assert list(out.columns) == ["Index", "demand"], "bad column names"
        assert np.isfinite(out["demand"]).all(), "non-finite predictions"
        assert ((out["demand"] >= 0) & (out["demand"] <= 1)).all(), "demand out of [0,1]"
        assert (out["Index"].to_numpy() == tec["Index"].to_numpy()).all(), "index order mismatch"
        path = f"submissions/{name}.csv"
        out.to_csv(path, index=False)
        print(f"wrote {path}  mean={preds.mean():.4f} min={preds.min():.4f} max={preds.max():.4f}")

    write("sub_lookup", pred_lookup)            # primary baseline candidate
    write("sub_lookup_nb", pred_lookup_nb)      # neighbor-fallback variant (A/B vs sub_lookup)
    write("sub_gbm", pred_gbm)
    write("sub_blend", pred_blend)


main()